# Check the TPU

In [1]:
import sys
import torch
import platform
import importlib.util

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("Platform:", platform.platform())

print("\nTPU package availability:")
print("torch_tpu:", importlib.util.find_spec("torch_tpu"))
print("torch_xla:", importlib.util.find_spec("torch_xla"))

print("\nPyTorch device information:")
print("CUDA available:", torch.cuda.is_available())

# Check the TPU device
try:
    import torch_tpu
    print("TorchTPU imported successfully")
except Exception as e:
    print("TorchTPU import:", repr(e))

try:
    import torch_xla
    print("PyTorch/XLA version:", torch_xla.__version__)
except Exception as e:
    print("PyTorch/XLA import:", repr(e))

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch version: 2.14.1+cpu
Platform: Linux-6.6.159+-x86_64-with-glibc2.39

TPU package availability:
torch_tpu: ModuleSpec(name='torch_tpu', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7a0205ff9eb0>, origin='/usr/local/lib/python3.13/dist-packages/torch_tpu/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/torch_tpu'])
torch_xla: None

PyTorch device information:
CUDA available: False
TorchTPU imported successfully
PyTorch/XLA import: ModuleNotFoundError("No module named 'torch_xla'")


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


## Verify actual TPU computation

In [2]:
import torch
import time

# Select TPU
device = torch.device("tpu")

# Create tensors directly on TPU
x = torch.randn(4, 4, device=device)
y = torch.randn(4, 4, device=device)

# Perform matrix multiplication
start_time = time.perf_counter()

z = x @ y

# Move result back to CPU
result = z.cpu()

elapsed_time = time.perf_counter() - start_time

print("TPU verification successful!")
print("--------------------------------")
print("Device:", z.device)
print("Result shape:", result.shape)
print("Result:")
print(result)
print("Execution time:", elapsed_time, "seconds")

TPU verification successful!
--------------------------------
Device: tpu:0
Result shape: torch.Size([4, 4])
Result:
tensor([[ 1.0953,  1.3098, -0.0283, -3.8731],
        [-0.6395,  1.6452,  0.8970,  1.5208],
        [-0.0174,  3.6909,  0.6147, -1.2500],
        [-4.8322, -3.0442,  0.2345,  5.0301]])
Execution time: 0.3522809129999587 seconds


## Check TPU properties

In [3]:
print("TPU availability:",
      torch.accelerator.is_available())

print("Current accelerator:",
      torch.accelerator.current_accelerator())

print("Number of TPU devices:",
      torch.accelerator.device_count())

print("TPU properties:")
print(torch.tpu.get_device_properties())

TPU availability: True
Current accelerator: tpu
Number of TPU devices: 1
TPU properties:
_TpuDeviceProperties(name='TPU v5e', total_memory=17179869184, multi_processor_count=1)


/tmp/ipykernel_3246/32538383.py:11: UserWarning: <TorchTpu/ApiStage/ExperimentalApi/> get_device_properties is experimental and subject to change: get_device_properties() is experimental and subject to change without notice.
  print(torch.tpu.get_device_properties())


## Transfer the dataset and model
We need these two files from your Windows project:
- data/processed/har_preprocessed.pt
- models/har_cnn_baseline.pth
We'll upload them to the Colab session.

In [6]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(filename)

Saving har_cnn_baseline.pth to har_cnn_baseline.pth

Uploaded files:
har_cnn_baseline.pth


## Verify the uploaded files

In [7]:
import os

for filename in [
    "har_preprocessed.pt",
    "har_cnn_baseline.pth"
]:
    print(filename, ":", os.path.exists(filename))

har_preprocessed.pt : True
har_cnn_baseline.pth : True


## Load the dataset and checkpoint

In [8]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time

# Load dataset
data = torch.load(
    "har_preprocessed.pt",
    map_location="cpu",
    weights_only=False
)

# Load trained model checkpoint
checkpoint = torch.load(
    "har_cnn_baseline.pth",
    map_location="cpu",
    weights_only=True
)

print("Dataset loaded successfully")
print("Dataset keys:", data.keys())

print("\nCheckpoint loaded successfully")
print("Architecture:", checkpoint["architecture"])
print("Input channels:", checkpoint["input_channels"])
print("Sequence length:", checkpoint["sequence_length"])
print("Number of classes:", checkpoint["num_classes"])
print("Original test accuracy:", checkpoint["test_accuracy"])

Dataset loaded successfully
Dataset keys: dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'train_indices', 'val_indices'])

Checkpoint loaded successfully
Architecture: HAR_CNN
Input channels: 6
Sequence length: 128
Number of classes: 6
Original test accuracy: 0.8920936545639634


## Reconstruct the trained model

In [9]:
import torch.nn as nn

class HAR_CNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        self.conv1 = nn.Conv1d(6, 32, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)

        return x


# Reconstruct model and load trained weights
model = HAR_CNN(num_classes=checkpoint["num_classes"])

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

total_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Model loaded successfully.")
print("Architecture:", checkpoint["architecture"])
print("Trainable parameters:", f"{total_parameters:,}")
print("Model mode:", "Evaluation" if not model.training else "Training")

Model loaded successfully.
Architecture: HAR_CNN
Trainable parameters: 274,342
Model mode: Evaluation


## Prepare the TPU test DataLoader

In [10]:
from torch.utils.data import DataLoader, TensorDataset

# Prepare test tensors on CPU
X_test_tensor = data["X_test"].detach().clone().to(torch.float32)
y_test_tensor = data["y_test"].detach().clone().to(torch.long)

# Create test dataset
BATCH_SIZE = 64

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inspect the dataset
signals, labels = next(iter(test_loader))

print("Test samples:", len(test_dataset))
print("Number of batches:", len(test_loader))
print("Batch size:", BATCH_SIZE)
print("Signal batch shape:", signals.shape)
print("Label batch shape:", labels.shape)
print("Signal device:", signals.device)
print("Label device:", labels.device)

Test samples: 2947
Number of batches: 47
Batch size: 64
Signal batch shape: torch.Size([64, 6, 128])
Label batch shape: torch.Size([64])
Signal device: cpu
Label device: cpu


## Check CNN compatibility with TPU

In [11]:
device = torch.device("tpu")

# Transfer model to TPU
model = model.to(device)
model.eval()

# Transfer one batch
signals, labels = next(iter(test_loader))
signals = signals.to(device)

# Run inference
with torch.inference_mode():
    outputs = model(signals)

print("TPU CNN test successful")
print("-------------------------")
print("Model device:", next(model.parameters()).device)
print("Input shape:", signals.shape)
print("Output shape:", outputs.shape)
print("Output device:", outputs.device)

TPU CNN test successful
-------------------------
Model device: tpu:0
Input shape: torch.Size([64, 6, 128])
Output shape: torch.Size([64, 6])
Output device: tpu:0


## TPU inference benchmark

In [12]:
# Select TPU
device = torch.device("tpu")

model = model.to(device)
model.eval()

NUM_RUNS = 3

def benchmark_tpu_inference(model, data_loader, device):

    model.eval()

    # ---------------------------------
    # 1. Full-dataset warm-up
    # ---------------------------------
    print("Running TPU warm-up...")

    with torch.inference_mode():
        for signals, _ in data_loader:
            signals = signals.to(device)
            _ = model(signals)

    # Wait for TPU operations to finish
    torch.accelerator.synchronize(device)

    print("Warm-up completed.")

    # ---------------------------------
    # 2. Actual benchmark
    # ---------------------------------
    run_times = []

    for run in range(NUM_RUNS):

        torch.accelerator.synchronize(device)
        start_time = time.perf_counter()

        with torch.inference_mode():
            for signals, _ in data_loader:
                signals = signals.to(device)
                _ = model(signals)

        # Wait until all TPU computations finish
        torch.accelerator.synchronize(device)

        elapsed_time = time.perf_counter() - start_time
        run_times.append(elapsed_time)

        print(f"Run {run + 1}: {elapsed_time:.4f} seconds")

    # ---------------------------------
    # 3. Calculate performance
    # ---------------------------------
    average_time = np.mean(run_times)

    return {
        "device": "tpu",
        "tpu_name": "Google TPU v5e",
        "average_time_seconds": average_time,
        "average_batch_time_ms": (
            average_time / len(data_loader) * 1000
        ),
        "throughput_samples_per_second": (
            len(data_loader.dataset) / average_time
        )
    }


tpu_results = benchmark_tpu_inference(
    model,
    test_loader,
    device
)

print("\nTPU Inference Benchmark Results")
print("--------------------------------")

for key, value in tpu_results.items():
    print(f"{key}: {value}")

Running TPU warm-up...
Warm-up completed.
Run 1: 0.0793 seconds
Run 2: 0.0798 seconds
Run 3: 0.0799 seconds

TPU Inference Benchmark Results
--------------------------------
device: tpu
tpu_name: Google TPU v5e
average_time_seconds: 0.07966398433336508
average_batch_time_ms: 1.6949783900715973
throughput_samples_per_second: 36992.877329206465


## Save the GPU benchmark results

In [13]:
tpu_results_df = pd.DataFrame([{
    **tpu_results,
    "tpu_memory_gb": 16,
    "pytorch_version": torch.__version__,
    "batch_size": BATCH_SIZE,
    "test_samples": len(test_dataset),
    "warmup_strategy": "full_dataset",
    "benchmark_runs": NUM_RUNS
}])

tpu_results_file = "inference_benchmark_tpu.csv"

tpu_results_df.to_csv(tpu_results_file, index=False)

display(tpu_results_df)

,device,tpu_name,average_time_seconds,average_batch_time_ms,throughput_samples_per_second,tpu_memory_gb,pytorch_version,batch_size,test_samples,warmup_strategy,benchmark_runs
0,tpu,Google TPU v5e,0.079664,1.694978,36992.877329,16,2.14.1+cpu,64,2947,full_dataset,3


In [14]:
from google.colab import files

files.download("inference_benchmark_tpu.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>